In [1]:
import os

In [2]:
%pwd

'e:\\Chicken_Disease_Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\Chicken_Disease_Classification'

In [5]:
# Entity
# Paste in \Chicken_Disease_Classification\entity\config_entity.py
# Model Trainer

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list


# We will use (prepare call backs)
@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path



In [6]:
from Chicken_Disease_Classification.constrants import * # Import Everything
from Chicken_Disease_Classification.utils.common import read_yaml,create_directories 
import tensorflow as tf

In [7]:
# Paste in Configuration (src\Chicken_Disease_Classification\config\configuration.py)



class ConfigurationManager: 
    
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):

            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root]) # Prepare Directories


    # We be use this during training
    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
          config=self.config.prepare_callbacks
          model_ckpt_dir=os.path.dirname(config.checkpoint_model_filepath)
          create_directories([
                Path(model_ckpt_dir),                     # Check Point Directory
                Path(config.tensorboard_root_log_dir)      # Tensorboard Directory
          ])

          prepare_callback_config=PrepareCallbacksConfig(                           # Convert Path to string
                root_dir=str(config.root_dir),
                tensorboard_root_log_dir=str(config.tensorboard_root_log_dir),
                checkpoint_model_filepath=str(config.checkpoint_model_filepath)
          )

          return prepare_callback_config
    
    
    
    def get_training_config(self) -> TrainingConfig:
          training=self.config.training           # From config
          prepare_base_model=self.config.prepare_base_model
          params=self.params   # Parameter.yaml
          training_data=os.path.join(self.config.data_ingestion.unzip_dir,"Chicken-fecal-images" )
          create_directories([Path(training.root_dir)])

          training_config=TrainingConfig(
                root_dir=Path(training.root_dir),
                trained_model_path=Path(training.trained_model_path),
                updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
                training_data=Path(training_data),
                params_epochs=params.EPOCHS,
                params_batch_size=params.BATCH_SIZE,
                params_is_augmentation=params.AUGMENTATION,
                params_image_size=params.IMAGE_SIZE

                
          )

          return training_config
    
    


            

In [8]:
# Components

import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time
from pathlib import Path

In [9]:
class PrepareCallbacks:
    def __init__(self,config:PrepareCallbacksConfig):
        self.config=config

    @property
    def _create_tb_callbacks(self):   # Create time stamp folder and create all the logs there
        timestamp=time.strftime("%y-%m-%d-%H-%M-%S")
        tb_running_log_dir=os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}",
        )
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)
    
    @property
    def _create_ckpt_callbacks(self):   # Create Check Point call backs
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True
        )
    
    def get_tb_ckpt_callbacks(self): # Get the check points along with time
        return[
            self._create_tb_callbacks,
            self._create_ckpt_callbacks
        ]

In [10]:
class Training:
    def __init__(self,config:TrainingConfig):
        self.config=config

    # Gettting the base model
    def get_base_model(self):   
        self.model=tf.keras.models.load_model(
            self.config.updated_base_model_path
        )


    # Divide the data set into train and test data with the help of train and validation generator
    def train_valid_generator(self):  

        datagenerator_kwargs=dict(
            rescale=1./255,     # Normalization
            validation_split=0.20
        )

        dataflow_kwargs=dict(
            target_size=self.config.params_image_size[:-1], # [224, 224]
            batch_size=self.config.params_batch_size,
            interpolation="bilinear" # Fill the gaps
        )

        valid_datagenerator=tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator=valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


        if self.config.params_is_augmentation:
            train_datagenerator=tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator=train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path:Path, model: tf.keras.Model):
        model.save(path)

    def train(self,callbacks_list:list):
        self.steps_per_epoch=self.train_generator.samples // self.train_generator.batch_size   # Convert to a whole number
        self.validation_steps=self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            callbacks=callbacks_list
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )



In [ ]:
# Pipeline

try:
    config=ConfigurationManager()    
    PrepareCallbacksConfig=config.get_prepare_callback_config()
    prepare_callbacks=PrepareCallbacks(config=PrepareCallbacksConfig)
    callback_list=prepare_callbacks.get_tb_ckpt_callbacks()

    training_config=config.get_training_config()
    training=Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(
        callbacks_list=callback_list
    )

except Exception as e :
    raise e

    
    

In [ ]:
# For seeing the log of tensor flow
# run :- tensorboard --logdir artifacts\prepare_callbacks
# run:-  tensorboard --logdir artifacts\prepare_callbacks\tensorboard_log_dir
# in terminal